In [0]:
from delta.tables import DeltaTable
from pyspark.sql import SparkSession
from datetime import datetime
from pyspark.sql import functions as F
import os

spark = SparkSession.builder.getOrCreate()

# Paths
raw_path = "abfss://raw@[YOUR_STORAGE_ACCOUNT_NAME].dfs.core.windows.net/"
bronze_delta_path = "abfss://processed@[YOUR_STORAGE_ACCOUNT_NAME].dfs.core.windows.net/bronze/"
csv_archive_path = "abfss://processed@[YOUR_STORAGE_ACCOUNT_NAME].dfs.core.windows.net/bronze_processed/"
quarantine_path = "abfss://processed@[YOUR_STORAGE_ACCOUNT_NAME].dfs.core.windows.net/quarantine/"

primary_keys = {
    "olist_customers_dataset": "customer_id",
    "olist_geolocation_dataset": ["geolocation_zip_code_prefix", "geolocation_lat", "geolocation_lng", "geolocation_city", "geolocation_state"],
    "olist_order_items_dataset": ["order_id", "order_item_id", "product_id"],
    "olist_order_payments_dataset": ["order_id", "payment_sequential"],
    "olist_order_reviews_dataset": ["review_id", "order_id"],
    "olist_orders_dataset": "order_id",
    "olist_products_dataset": "product_id",
    "olist_sellers_dataset": "seller_id",
    "product_category_name_translation": "product_category_name"
}

# List CSVs
files = [f.path for f in dbutils.fs.ls(raw_path) if f.name.endswith(".csv")]

for file in files:
    # Get table name from filename
    file_name = os.path.basename(file)
    table_name = file_name.replace(".csv", "")
    run_timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

    if table_name not in primary_keys:
        print(f"--> [Quarantine] No configuration found for: {table_name}")
        target_quarantine = os.path.join(quarantine_path, f"{run_timestamp}_{file_name}")
        dbutils.fs.mv(file, target_quarantine)
        print(f"--> Moved {file_name} to quarantine folder.")
        continue

    print(f"--> [START] Processing: {table_name}")

    # Read with robust CSV settings for Olist's messy strings
    df_raw = (spark.read
              .format("csv")
              .option("header", "true")
              .option("inferSchema", "false")
              .option("multiLine", "true")
              .option("escape", '"')
              .option("quote", '"')
              .load(file))
    
    # Adding metadata for SCD tracking
    df_bronze = df_raw.withColumn("_ingested_at", F.current_timestamp())

    # Pk Integrity Isolation
    pk = primary_keys[table_name]
    pk_cols = pk if isinstance(pk, list) else [pk]
    null_pk_condition = " OR ".join([f"{col} IS NULL" for col in pk_cols])

    # Split into valid and invalid records
    df_invalid = df_bronze.filter(null_pk_condition)
    df_valid = df_bronze.filter(f"NOT ({null_pk_condition})")

    if not df_invalid.isEmpty():
        print(f"--> Null PKs found in {table_name}")
        rejected_rows_path = os.path.join(quarantine_path, "rejected", table_name)
        df_to_quarantine = (df_invalid
                            .withColumn("rejected_at", F.current_timestamp())
                            .withColumn("_original_source_file", F.lit(file_name)))
        quarantine_table_path = os.path.join(quarantine_path, "rejected_records", table_name)
        (df_to_quarantine.write.format("delta").mode("append").option("mergeSchema", "false").save(quarantine_table_path))

    # Define unique path for this specific table
    target_table_path = os.path.join(bronze_delta_path, table_name)
    (df_valid.write.format("delta").mode("append").save(target_table_path))

    # Orchestration: Move file to 'bronze processed container' only after Spark succeeds
    final_csv_destination = os.path.join(csv_archive_path, file_name)
    dbutils.fs.mv(file, final_csv_destination)
    print(f"--> [SUCCESS] Data appended to Delta table and CSV moved to archive.\n")

message = f"Bronze Ingestion Completed"
dbutils.notebook.exit(message)